In [1]:
import argparse
import json
import os
import random
import re
import shutil
from collections import OrderedDict, defaultdict
from functools import partial
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
import torch
import tqdm


from sklearn.model_selection import GroupKFold
from torch import nn
from torch.nn import functional as F

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import RobertaConfig, RobertaModel, RobertaTokenizer, AutoConfig, AutoModel, AutoTokenizer
from transformers.optimization import (AdamW, get_cosine_schedule_with_warmup,
                                       get_linear_schedule_with_warmup,
                                       get_cosine_with_hard_restarts_schedule_with_warmup)

2026-04-28 09:15:04.652919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777367704.663318      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777367704.666418      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 09:15:04.683771: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

ImportError: cannot import name 'AdamW' from 'transformers.optimization' (/usr/local/lib/python3.11/dist-packages/transformers/optimization.py)

In [2]:
from shutil import copyfile
copyfile(src = "../input/utils-v10/utilsv10.py", dst = "../working/utilsv10.py")
copyfile(src = "../input/utils-v10/dataset10.py", dst = "../working/dataset10.py")

FileNotFoundError: [Errno 2] No such file or directory: '../input/utils-v10/utilsv10.py'

In [3]:
from utilsv10 import (binary_focal_loss, get_learning_rate, jaccard_list, get_best_pred, ensemble, ensemble_words,
                   load_model, save_model, set_seed, write_event, evaluate, get_predicts_from_token_logits, map_to_word)

ModuleNotFoundError: No module named 'utilsv10'

In [4]:
from dataset10 import TrainDataset, MyCollator

ModuleNotFoundError: No module named 'dataset10'

## Parse data

In [5]:
test = pd.read_csv('../input/tweet-sentiment-extraction/test.csv')

In [6]:
tokenizer = AutoTokenizer.from_pretrained('../input/roberta-base/', do_lower_case=False)

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../input/roberta-base/'. Use `repo_type` argument if needed.

In [7]:
class Args:
    post = False
    tokenizer = tokenizer
    offset = 4
    batch_size = 16
    workers = 1
args = Args()

NameError: name 'tokenizer' is not defined

## dataset

In [8]:
collator = MyCollator()
test_set = TrainDataset(test, None, tokenizer=tokenizer, mode='test', offset=args.offset)
test_loader = DataLoader(test_set, batch_size=args.batch_size, shuffle=False, collate_fn=collator,
                                 num_workers=args.workers)

NameError: name 'MyCollator' is not defined

## model

In [9]:
class TweetModel(nn.Module):

    def __init__(self, pretrain_path=None, dropout=0.2, config=None):
        super(TweetModel, self).__init__()
        if config is not None:
            self.bert = AutoModel.from_config(config)
        else:
            config = AutoConfig.from_pretrained(pretrain_path, output_hidden_states=True)
            self.bert = AutoModel.from_pretrained(
                pretrain_path, cache_dir=None, config=config)
        
        self.cnn =  nn.Conv1d(self.bert.config.hidden_size*3, self.bert.config.hidden_size, 3, padding=1)

        # self.rnn = nn.LSTM(self.bert.config.hidden_size, self.bert.config.hidden_size//2, num_layers=2,
        #                     batch_first=True, bidirectional=True)
        self.gelu = nn.GELU()

        self.whole_head = nn.Sequential(OrderedDict([
            ('dropout', nn.Dropout(0.1)),
            ('l1', nn.Linear(self.bert.config.hidden_size*3, 256)),
            ('act1', nn.GELU()),
            ('dropout', nn.Dropout(0.1)),
            ('l2', nn.Linear(256, 2))
        ]))
        self.se_head = nn.Linear(self.bert.config.hidden_size, 2)
        self.inst_head = nn.Linear(self.bert.config.hidden_size, 2)
        self.dropout = nn.Dropout(0.1)

    def forward(self, inputs, masks, token_type_ids=None, input_emb=None):
        _, pooled_output, hs = self.bert(
            inputs, masks, token_type_ids=token_type_ids, inputs_embeds=input_emb)

        seq_output = torch.cat([hs[-1],hs[-2],hs[-3]], dim=-1)

        # seq_output = hs[-1]

        avg_output = torch.sum(seq_output*masks.unsqueeze(-1), dim=1, keepdim=False)
        avg_output = avg_output/torch.sum(masks, dim=-1, keepdim=True)
        # +max_output
        whole_out = self.whole_head(avg_output)

        seq_output = self.gelu(self.cnn(seq_output.permute(0,2,1)).permute(0,2,1))
        
        se_out = self.se_head(self.dropout(seq_output))  #()
        inst_out = self.inst_head(self.dropout(seq_output))
        return whole_out, se_out[:, :, 0], se_out[:, :, 1], inst_out

In [10]:
def predict(model: nn.Module, valid_df, valid_loader, args, progress=False) -> Dict[str, float]:
    # run_root = Path('../experiments/' + args.run_root)
    model.eval()
    all_end_pred, all_whole_pred, all_start_pred, all_inst_out = [], [], [], []
    if progress:
        tq = tqdm.tqdm(total=len(valid_df))
    with torch.no_grad():
        for tokens, types, masks, _, _, _, _, _, _, _ in valid_loader:
            if progress:
                batch_size = tokens.size(0)
                tq.update(batch_size)
            masks = masks.cuda()
            tokens = tokens.cuda()
            types = types.cuda()
            whole_out, start_out, end_out, inst_out = model(tokens, masks, types)
            start_out = start_out.masked_fill(~masks.bool(), -1000)
            end_out= end_out.masked_fill(~masks.bool(), -1000)   

            start_out = torch.softmax(start_out, dim=-1)
            end_out = torch.softmax(end_out, dim=-1)

            all_whole_pred.append(torch.softmax(whole_out, dim=-1)[:,1].cpu().numpy())
            inst_out = torch.softmax(inst_out, dim=-1)
            for idx in range(len(start_out)):
                length = torch.sum(masks[idx,:]).item()-1 # -1 for last token
                all_start_pred.append(torch.softmax(start_out[idx, args.offset:length], axis=-1).cpu())
                all_end_pred.append(torch.softmax(end_out[idx, args.offset:length], axis=-1).cpu())
                all_inst_out.append(inst_out[idx,:,1].cpu())
            assert all_start_pred[-1].dim()==1

    all_whole_pred = np.concatenate(all_whole_pred)
    
    if progress:
        tq.close()
    return all_whole_pred, all_start_pred, all_end_pred, all_inst_out

## predict

In [11]:
config = RobertaConfig.from_pretrained('../input/roberta-base', output_hidden_states=True)
model = TweetModel(config=config)

# load_model(model, '../input/large-uncased-wwm-weights/best-model-0.pt')

# model.cuda()
# all_senti_preds, all_start_preds, all_end_preds = predict(model, test, test_loader, None)
# preds = get_predicts(all_senti_preds, all_start_preds, all_end_preds, test)



OSError: Can't load the configuration of '../input/roberta-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure '../input/roberta-base' is the correct path to a directory containing a config.json file

In [12]:
all_whole_preds, all_start_preds, all_end_preds, all_inst_preds = [], [], [], []

    
for fold in range(5):
    load_model(model, '../input/roberta-weights-v10/best-model-%d.pt' % fold)
    model.cuda()
    fold_whole_preds, fold_start_preds, fold_end_preds, fold_inst_preds = predict(model, test, test_loader, args, progress=True)

    all_whole_preds.append(fold_whole_preds)
    all_start_preds.append(fold_start_preds)
    all_end_preds.append(fold_end_preds)
    all_inst_preds.append(fold_inst_preds)


all_whole_preds, all_start_preds, all_end_preds, all_inst_preds = ensemble(all_whole_preds, all_start_preds, all_end_preds, all_inst_preds, test)
word_preds, inst_word_preds, scores = get_predicts_from_token_logits(all_whole_preds, all_start_preds, all_end_preds, all_inst_preds, test, args)
# word_preds, inst_word_preds, scores = get_predicts_from_token_logits(fold_whole_preds, fold_start_preds, fold_end_preds, fold_inst_preds, test, args)



NameError: name 'load_model' is not defined

In [13]:
test['selected_text'] = word_preds
# test.loc[replace_idx, 'selected_text'] = test.loc[replace_idx, 'text']
def f(selected):
    return " ".join(set(selected.lower().split()))
# test.selected_text = test.selected_text.map(f)
test[['textID','selected_text']].to_csv('submission.csv', index=False)

NameError: name 'word_preds' is not defined

In [14]:
test.head(20)

,textID,text,sentiment
0,80a1e6bc32,I just saw a shooting star... I made my wish,positive
1,863097735d,gosh today sucks! i didnt get my tax returns! ...,negative
2,264cd5277f,tired and didn`t really have an exciting Satur...,neutral
3,baee1e6ffc,i`ve been eating cheetos all morning..,neutral
4,67d06a8dee,haiiii sankQ i`m fineee ima js get a checkup ...,neutral
5,7438c9c09a,CONGRATS on graduating college!,positive
6,77a6add433,"loved, but hated driving in pollution",positive
7,0f5c867e5d,In weho! They`re are playing a lot of brit,neutral
8,02d47f1829,_NJ Oh! I`m only in my 7 I just joined Twitt...,neutral
9,7b46dad07f,awww wish I was there! Have a brew for me B!,positive
